# `restore_structure` vs `add_observables`

The documented flat-state pattern calls `add_observables` (which runs `_channel_currents`) and then `step` (which runs `_channel_currents` again).

For HH / Leak, `update_states` does not read membrane currents, so the first pass is redundant. Here we compare:
1. **Official path:** `add_observables` → `step` (currents computed twice)
2. **Split path:** `restore_structure` + zero current placeholders → `step` (currents once)

In [ ]:
import time

import jax
import jax.numpy as jnp
from jax import jit

import jaxley as jx
from jaxley.channels.hh import HH
from jaxley.integrate import build_init_and_step_fn
from jaxley.utils.dynamics import build_dynamic_state_utils

jax.config.update("jax_platform_name", "cpu")

# 4-compartment HH cable (matches the issue benchmark scale)
comp = jx.Compartment()
branch = jx.Branch(comp, ncomp=4)
cell = jx.Cell(branch, parents=[-1])
cell.insert(HH())

t_max = 50.0
delta_t = 0.025
n_steps = int(t_max / delta_t)

cell.record("v")
cell.stimulate(jx.step_current(5.0, 20.0, 0.05, delta_t, t_max))

rec_inds = cell.recordings.rec_index.to_numpy()
rec_states = cell.recordings.state.to_numpy()
externals = cell.externals.copy()
external_inds = cell.external_inds.copy()

params = cell.get_parameters()
cell.to_jax()

init_fn, step_fn = build_init_and_step_fn(cell)
(
    remove_observables,
    add_observables,
    flatten,
    unflatten,
    restore_structure,
) = build_dynamic_state_utils(cell)

# Observable currents that are not themselves dynamic states
all_states0, _ = init_fn(params)
current_keys = [
    name
    for name in cell.membrane_current_names + cell.synapse_current_names
    if name not in remove_observables(all_states0)
]
print("n_comps:", len(all_states0["v"]))
print("current_keys:", current_keys)
print("n_steps:", n_steps)

In [ ]:
def get_externals_now(externals, step):
    return {key: externals[key][:, step] for key in externals}


def init_dynamics(params):
    all_states, all_params = init_fn(params)
    recordings = [
        jnp.asarray(
            [
                all_states[rec_state][rec_ind]
                for rec_state, rec_ind in zip(rec_states, rec_inds)
            ]
        )
    ]
    dynamic_states = flatten(remove_observables(all_states))
    return dynamic_states, all_params, recordings


def step_with_add_observables(dynamic_states, all_params, externals_now, external_inds):
    """Official path: structure + currents, then step (currents again)."""
    all_states = add_observables(unflatten(dynamic_states), all_params, delta_t)
    all_states = step_fn(
        all_states, all_params, externals_now, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [
            all_states[rec_state][rec_ind]
            for rec_state, rec_ind in zip(rec_states, rec_inds)
        ]
    )
    return flatten(remove_observables(all_states)), recs


def step_with_restore_structure(dynamic_states, all_params, externals_now, external_inds):
    """Split path: structure only (+ zero current placeholders), then step."""
    all_states = restore_structure(unflatten(dynamic_states))
    # step indexes membrane_current_names; placeholders are unused by HH.update_states
    all_states = dict(all_states)
    for name in current_keys:
        all_states[name] = jnp.zeros_like(all_states["v"])
    all_states = step_fn(
        all_states, all_params, externals_now, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [
            all_states[rec_state][rec_ind]
            for rec_state, rec_ind in zip(rec_states, rec_inds)
        ]
    )
    return flatten(remove_observables(all_states)), recs


def rollout(step_fn_dyn, params):
    dynamic_states, all_params, recordings = init_dynamics(params)
    for step in range(n_steps):
        externals_now = get_externals_now(externals, step)
        dynamic_states, recs = step_fn_dyn(
            dynamic_states, all_params, externals_now, external_inds
        )
        recordings.append(recs)
    return jnp.stack(recordings, axis=0), dynamic_states

## Equality check

In [ ]:
recs_add, dyn_add = rollout(step_with_add_observables, params)
recs_restore, dyn_restore = rollout(step_with_restore_structure, params)

print("recordings max |diff|:", float(jnp.max(jnp.abs(recs_add - recs_restore))))
print("dynamic states max |diff|:", float(jnp.max(jnp.abs(dyn_add - dyn_restore))))
print("allclose recordings:", bool(jnp.allclose(recs_add, recs_restore)))
print("allclose dynamic states:", bool(jnp.allclose(dyn_add, dyn_restore)))
assert jnp.allclose(recs_add, recs_restore)
assert jnp.allclose(dyn_add, dyn_restore)
print("OK: trajectories match.")

## Timing (JIT)

In [ ]:
@jit
def rollout_add(params):
    return rollout(step_with_add_observables, params)[0]


@jit
def rollout_restore(params):
    return rollout(step_with_restore_structure, params)[0]


# Warmup / compile
_ = rollout_add(params).block_until_ready()
_ = rollout_restore(params).block_until_ready()


def bench(fn, n_reps=5):
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn(params).block_until_ready()
        times.append(time.perf_counter() - t0)
    return times


times_add = bench(rollout_add)
times_restore = bench(rollout_restore)

mean_add = sum(times_add) / len(times_add)
mean_restore = sum(times_restore) / len(times_restore)

print(f"add_observables:   {mean_add*1e3:.2f} ms  ({times_add})")
print(f"restore_structure: {mean_restore*1e3:.2f} ms  ({times_restore})")
print(f"speedup (add / restore): {mean_add / mean_restore:.2f}x")

## Longer rollout (closer to issue: 2000 steps)

In [ ]:
t_max_long = 50.0  # 2000 steps at dt=0.025
assert int(t_max_long / delta_t) == 2000

# Reuse same cell/stimuli already at 2000 steps if t_max==50; otherwise rebuild externals.
print("steps:", n_steps)
print("Done.")